In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers scikit-learn lxml
!pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.5 MB/s eta 0:00:00:00:0100:01


In [2]:
import os

INPUT_FILE  = "/kaggle/input/datasets/moulighosh28/hallucination-classifier/hallucination_dataset2 (4).npz"
save_file = "/kaggle/working/hallucination_dataset2.npz"

In [3]:
from datasets import load_dataset

data = []

dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards")

train_data = dataset["train"]
print("Total rows in dataset:", len(train_data))

print("Columns:", train_data.column_names)

subset = train_data.select(range(12000, 25000))

prev_question = None   # keep track of previous question

for row in subset:

    question = row["input"].strip()
    answer = row["output"].strip()

    # skip if same as previous question
    if question == prev_question:
        continue

    if question and answer:
        data.append((question, answer))
        prev_question = question   # update previous question

print("Total valid Q/A pairs:", len(data))

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Total rows in dataset: 33955
Columns: ['input', 'output', 'instruction']
Total valid Q/A pairs: 12644


In [4]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

# Quantization config (NEW WAY)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    output_hidden_states=True,
    output_attentions=True
)

model.eval()


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['output_attentions', 'output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [5]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#  Similarity model
sim_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)  #SBERT

#  NLI model
nli_name = "roberta-large-mnli"

nli_tokenizer = AutoTokenizer.from_pretrained(nli_name)

nli_model = AutoModelForSequenceClassification.from_pretrained(nli_name)
nli_model.to(device)
nli_model.eval()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
 

In [6]:
def compute_entropy(probs):
    probs = torch.clamp(probs, min=1e-9)
    return -(probs * torch.log(probs)).sum(dim=-1)


def compute_similarity(ref, gen):
    with torch.no_grad():
        emb = sim_model.encode([ref, gen], convert_to_tensor=True)
        score = util.cos_sim(emb[0], emb[1])
    return score.item()


def run_nli(premise, hypothesis):
    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(nli_model.device)

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)
    labels = ["contradiction", "neutral", "entailment"]

    return labels[torch.argmax(probs, dim=-1).item()]

In [7]:
def extract_first_token_features(prompt, model, tokenizer, top_k=20):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(model.device)

    input_ids = inputs["input_ids"]
    start_pos = input_ids.shape[1]

    # ── First forward pass — get next token ──────────────────
    with torch.no_grad():
        outputs    = model(**inputs)
        logits     = outputs.logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1).unsqueeze(-1)

    full_input_ids = torch.cat([input_ids, next_token], dim=1)

    # ── Second forward pass — get internals ──────────────────
    with torch.no_grad():
        outputs = model(
            input_ids=full_input_ids,
            output_hidden_states=True,
            output_attentions=True
        )

    # 1. SOFTMAX FEATURES ─────────────────────────────────────
    logits_first     = outputs.logits[:, start_pos - 1, :]
    probs            = torch.softmax(logits_first, dim=-1)
    topk_vals, _     = torch.topk(probs, k=top_k, dim=-1)
    softmax_features = topk_vals.squeeze(0).float().cpu().numpy()
    # shape: [20] ✅

    # 2. ATTENTION FEATURES ───────────────────────────────────
    MAX_INPUT_LEN = 64
    attn_last = outputs.attentions[-1]  # [1, num_heads, seq_len, seq_len]
    num_heads = attn_last.shape[1]      # 32 for Mistral-7B

    # attention row of first generated token over all input tokens
    attn_row   = attn_last[0, :, start_pos, :start_pos]
    # shape: [32, actual_input_len]

    actual_len = min(attn_row.shape[1], MAX_INPUT_LEN)

    # Padded — fixed size, created on same device as model
    attn_fixed = torch.zeros(num_heads, MAX_INPUT_LEN,
                             dtype=torch.float32,
                             device=attn_last.device)  # ← same device ✅
    attn_fixed[:, :actual_len] = attn_row[:, :actual_len].float()
    attn_padded = attn_fixed.cpu().numpy().flatten()
    # shape: [2048] ✅

    # Raw — variable size for flexible later use
    attn_raw = attn_row.detach().float().cpu().numpy().flatten()
    # shape: [32 * actual_len] — variable ✅

    # 3. FFN FEATURES ─────────────────────────────────────────
    last_layer        = model.model.layers[-1]
    hidden_before_ffn = outputs.hidden_states[-2][:, start_pos, :]
    normed_hidden     = last_layer.post_attention_layernorm(hidden_before_ffn)
    gate              = last_layer.mlp.gate_proj(normed_hidden)
    up                = last_layer.mlp.up_proj(normed_hidden)
    ffn_activation    = torch.nn.functional.silu(gate) * up
    ffn_vector        = ffn_activation.squeeze(0).detach().float().cpu().numpy()
    # shape: [14336] ✅

    return softmax_features, attn_padded, attn_raw, ffn_vector

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"# Initialize
import gc
from tqdm import tqdm
session_count = 0
X_softmax     = []
X_attn_padded = []  # fixed [2048]
X_attn_raw    = []  # variable
X_ffn         = []
labels        = []

# Load checkpoint
if os.path.exists(INPUT_FILE):
    saved = np.load(INPUT_FILE, allow_pickle=True)
    X_softmax     = list(saved["X_softmax"])
    X_attn_padded = list(saved["X_attn_padded"])
    X_attn_raw    = list(saved["X_attn_raw"])
    X_ffn         = list(saved["X_ffn"])
    labels        = list(saved["y"])
    start_index = int(saved["last_index"]) + 1
    print(f"Resumed from checkpoint: {start_index} samples")
else:
    start_index = 0

print("Loaded samples:", len(labels))
tokenizer.pad_token = tokenizer.eos_token

# Main loop
for i in tqdm(range(start_index, len(data))):
    session_count += 1

    if i % 50 == 0:
        gc.collect()
        torch.cuda.empty_cache()

    question, reference = data[i]
    prompt = f"Answer the following question in one short sentence \nQuestion: {question}\nAnswer:"

    # Extract features
    with torch.no_grad():
        softmax_f, attn_padded_f, attn_raw_f, ffn_f = \
            extract_first_token_features(prompt, model, tokenizer)

    # Generate full answer for labeling
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(model.device)

    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            use_cache=False
        )

    input_len  = inputs["attention_mask"].sum().item()
    gen_tokens = gen_ids[0][int(input_len):]
    generated  = tokenizer.decode(gen_tokens, skip_special_tokens=True)



    # Labeling
    similarity = compute_similarity(reference, generated)
    nli_result = run_nli(reference, generated)

    # Add safety check: if NLI and similarity strongly disagree, skip or flag for review
    if nli_result == "entailment" and similarity < 0.60:
        del inputs, gen_ids, gen_tokens
        del softmax_f, attn_padded_f, attn_raw_f, ffn_f  # ✅
        gc.collect()
        torch.cuda.empty_cache()  
        continue  # or label = -1 for manual review
    elif nli_result == "contradiction" and similarity > 0.85: 
        del inputs, gen_ids, gen_tokens
        del softmax_f, attn_padded_f, attn_raw_f, ffn_f  # ✅
        gc.collect()
        torch.cuda.empty_cache()
        continue
    elif nli_result == "entailment":
        label = 0
    elif nli_result == "contradiction":
        label = 1
    elif similarity >= 0.91:
        label = 0
    elif similarity <= 0.40:
        label = 1
    else:
        del inputs, gen_ids, gen_tokens
        del softmax_f, attn_padded_f, attn_raw_f, ffn_f  # ✅
        gc.collect()
        torch.cuda.empty_cache()
        continue

   



    # Append
    X_softmax.append(softmax_f)
    X_attn_padded.append(attn_padded_f)  # [2048]
    X_attn_raw.append(attn_raw_f)        # variable
    X_ffn.append(ffn_f)
    labels.append(label)

    if session_count < 10 :
        print("\nQUESTION:", question)
        print("REFERENCE:", reference)
        print("GENERATED:", generated)
        print("-" * 10)
        print("Similarity:", similarity)
        print("NLI:", nli_result)
        print("Assigned label:", label)
        print("-" * 40)

    if len(labels) % 50 == 0:
        print("\nQUESTION:", question)
        print("REFERENCE:", reference)
        print("GENERATED:", generated)
        print("-" * 10)
        print("Similarity:", similarity)
        print("NLI:", nli_result)
        print("Assigned label:", label)
        print("-" * 40)

    # Cleanup
    del inputs, gen_ids, gen_tokens
    del softmax_f, attn_padded_f, attn_raw_f, ffn_f
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    # Checkpoint every 20 samples
    if len(labels) % 20 == 0:
        np.savez(
            save_file,
            X_softmax     = np.array(X_softmax),
            X_attn_padded = np.array(X_attn_padded),
            X_attn_raw    = np.array(X_attn_raw, dtype=object),
            X_ffn         = np.array(X_ffn),
            y             = np.array(labels),
            last_index    = i
        )
        print("Checkpoint saved at", len(labels))

# Final save
y = np.array(labels)
print("Label 0:", np.sum(y == 0))
print("Label 1:", np.sum(y == 1))
print("Final dataset size:", len(y))

np.savez(
    save_file,
    X_softmax     = np.array(X_softmax),
    X_attn_padded = np.array(X_attn_padded),
    X_attn_raw    = np.array(X_attn_raw, dtype=object),
    X_ffn         = np.array(X_ffn),
    y             = y,
    last_index    = i
)
print("Final dataset saved.")

Resumed from checkpoint: 9065 samples
Loaded samples: 5520



  0%|          | 0/3579 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  0%|          | 1/3579 [00:21<21:11:59, 21.33s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  0%|          | 2/3579 [00:38<18:42:30, 18.83s/it]


QUESTION: What are some of the nutrients that can be malabsorbed in cases of pancreatic insufficiency?
REFERENCE: Pancreatic insufficiency can lead to malabsorption of several nutrients, including fat, fat-soluble vitamins (such as vitamins A, D, E, and K), and vitamin B12. This is because the pancreas produces enzymes that are necessary for the proper digestion and absorption of these nutrients. When the pancreas is unable to produce enough enzymes, these nutrients may not be properly broken down and absorbed in the small intestine, leading to malabsorption. Malabsorption of these nutrients can result in a range of symptoms, including diarrhea, weight loss, and deficiencies in these essential vitamins and minerals.
GENERATED: In cases of pancreatic insufficiency, nutrients like fats, proteins, and certain vitamins (A, D, E, and K) can be malabsorbed.
----------
Similarity: 0.8198359608650208
NLI: entailment
Assigned label: 0
----------------------------------------


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  0%|          | 3/3579 [00:53<17:11:33, 17.31s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  0%|          | 4/3579 [01:09<16:42:09, 16.82s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  0%|          | 5/3579 [01:23<15:24:41, 15.52s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



QUESTION: What is the term used to describe the condition in which pancreatic beta cells are unable to sustain a state of increased insulin production in type 2 diabetes mellitus?
REFERENCE: The term used to describe this condition is "β cell burnout". In type 2 diabetes mellitus, the body becomes resistant to the effects of insulin, which means that the pancreas must produce more insulin in order to maintain normal blood sugar levels. Over time, this increased demand for insulin can lead to exhaustion and dysfunction of the pancreatic beta cells, which are responsible for producing insulin. This condition is known as β cell burnout, and can contribute to the progression of type 2 diabetes and its associated complications. To prevent or manage type 2 diabetes, it is important to maintain a healthy lifestyle, including regular exercise, a balanced diet, and appropriate medical treatment as needed.
GENERATED: Beta-cell dysfunction or failure.
----------
Similarity: 0.6362159252166748
NL

...

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  6%|▋         | 226/3579 [55:49<12:48:15, 13.75s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  6%|▋         | 227/3579 [56:06<13:27:53, 14.46s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  6%|▋         | 228/3579 [56:16<12:27:55, 13.39s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  6%|▋         | 229/3579 [56:34<13:40:58, 14.70s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  6%|▋         | 230/3579 [56:45<12:31:07, 13.46s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  6%|▋         | 231/3579 [56:56<11:49:23, 12.71s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  6%|▋         | 232/3579 [57:13<13:03:45, 14.05s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  7%|▋         | 233/3579 [57:27<13:07:36, 14.12s/it]Setting `pad_token_id` t

Checkpoint saved at 5660


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  7%|▋         | 243/3579 [1:00:43<16:05:34, 17.37s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  7%|▋         | 244/3579 [1:00:57<15:08:12, 16.34s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  7%|▋         | 245/3579 [1:01:08<13:37:10, 14.71s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  7%|▋         | 246/3579 [1:01:18<12:21:35, 13.35s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  7%|▋         | 247/3579 [1:01:28<11:12:41, 12.11s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  7%|▋         | 248/3579 [1:01:30<8:35:04,  9.28s/it] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  7%|▋         | 249/3579 [1:01:36<7:35:24,  8.21s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  7%|▋         | 250/3579 [1:01:51<9:37:10, 10.40s/it]Setting `p

Checkpoint saved at 5680


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

# ---------------------------------
# Load from checkpoint
# ---------------------------------

save_file = "/content/drive/MyDrive/hallucination_dataset1.npz"

saved         = np.load(save_file, allow_pickle=True)
X_softmax     = np.array(list(saved["X_softmax"]))
X_attn_padded = np.array(list(saved["X_attn_padded"]))
X_ffn         = np.array(list(saved["X_ffn"]))
y             = np.array(list(saved["y"]))

print(f"Loaded {len(y)} samples")
print(f"Label 0 (non-hallucinated): {np.sum(y == 0)}")
print(f"Label 1 (hallucinated):     {np.sum(y == 1)}")

# ---------------------------------
# SPLIT (same indices for all)
# ---------------------------------

indices = np.arange(len(y))

train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42
)

def prepare_data(X, scaler_type="standard"):
    X_train = X[train_idx]
    X_test  = X[test_idx]

    if scaler_type == "minmax":
        scaler = MinMaxScaler()
    else:
        scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    return X_train, X_test

# Prepare each feature set
X_softmax_train, X_softmax_test = prepare_data(X_softmax,     "standard")
X_attn_train,    X_attn_test    = prepare_data(X_attn_padded, "minmax")
X_ffn_train,     X_ffn_test     = prepare_data(X_ffn,         "standard")

y_train = y[train_idx]
y_test  = y[test_idx]

# ---------------------------------
# Device
# ---------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def to_tensor(X, y):
    return (
        torch.tensor(X, dtype=torch.float32).to(device),
        torch.tensor(y, dtype=torch.long).to(device)
    )

X_softmax_train, y_train_t = to_tensor(X_softmax_train, y_train)
X_softmax_test,  y_test_t  = to_tensor(X_softmax_test,  y_test)

X_attn_train, _ = to_tensor(X_attn_train, y_train)
X_attn_test,  _ = to_tensor(X_attn_test,  y_test)

X_ffn_train, _  = to_tensor(X_ffn_train,  y_train)
X_ffn_test,  _  = to_tensor(X_ffn_test,   y_test)

# ---------------------------------
# Class weights
# ---------------------------------

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

# ---------------------------------
# Model
# ---------------------------------

class Classifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

# ---------------------------------
# Train function
# ---------------------------------

def train_model(X_train, y_train, input_dim):
    model     = Classifier(input_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    batch_size = 128
    epochs     = 50

    for epoch in range(epochs):
        model.train()
        perm       = torch.randperm(X_train.size(0))
        epoch_loss = 0

        for i in range(0, X_train.size(0), batch_size):
            idx     = perm[i:i + batch_size]
            x_batch = X_train[idx]
            y_batch = y_train[idx]

            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Loss: {epoch_loss:.4f}")

    return model

# ---------------------------------
# Train 3 classifiers
# ---------------------------------

print("\nTraining SOFTMAX model")
clf_softmax = train_model(X_softmax_train, y_train_t, X_softmax_train.shape[1])

print("\nTraining ATTENTION model")
clf_attn = train_model(X_attn_train, y_train_t, X_attn_train.shape[1])

print("\nTraining FFN model")
clf_ffn = train_model(X_ffn_train, y_train_t, X_ffn_train.shape[1])

# ---------------------------------
# Evaluation (ensemble)
# ---------------------------------

def get_probs(model, X):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(X), dim=1)[:, 1]

p_softmax = get_probs(clf_softmax, X_softmax_test)
p_attn    = get_probs(clf_attn,    X_attn_test)
p_ffn     = get_probs(clf_ffn,     X_ffn_test)

# Weighted ensemble
probs = (
    0.2 * p_softmax +
    0.3 * p_attn    +
    0.5 * p_ffn
)

roc   = roc_auc_score(y_test, probs.cpu())
preds = (probs > 0.5).cpu()
f1    = f1_score(y_test, preds)

print("\n--- Results ---")
print(f"ROC-AUC:  {roc:.4f}")
print(f"F1 Score: {f1:.4f}")